# Experiment 1: perception (state and element recognition)

Evaluates VLM perception on the UFV campus dataset. Three perception questions
across stairs, elevators and facility signs:

| Question | Categories | n | Task type |
|-|-|-|-|
| `element_identification` | stairs + elevators + facility_signs | 34 | Multi-class (stairs, elevator, bathroom) |
| `stairs_direction` | stairs | 15 | Binary, up or down (8/7) |
| `elevator_state` | elevators | 10 | Binary, open or closed (5/5) |

## Models

Loaded from `models.json`: PaliGemma 2 (several variants) and Gemma 4 (E2B, E4B),
dispatched by family detection. Run on a Colab A100.

## Metrics (per question, per model)

- Accuracy: fraction correct.
- Macro precision, recall and F1: averaged over classes with at least 5 instances.
- Keyword recall (`element_identification` only): fraction of images whose response contains an expected keyword.
- Confusion matrix and average output length.

Aggregated per model: overall accuracy, state-recognition macro P/R/F1 (stairs and
elevator state combined), and operational metrics (inference time, VRAM peak,
throughput, load time).

## Inputs at `/content/`

- `images.zip`: unpacks to `/content/images/<category>/`.
- `annotations.zip`: unpacks to `/content/annotations/{evaluation_prompts.json, ground_truth.json}`.
- `models.json`: copy of `config/models.json`.

## Outputs

```
/content/results/
  <model>/
    results.csv      one row per (image, question)
    aggregate.csv    per-question aggregate metrics
    efficiency.json  operational and aggregated metrics
    matrices.json    per-question confusion matrices and per-class P/R/F1
  all_models_summary.csv
```

## 1. Install dependencies

On a fresh runtime, run this cell, then Runtime -> Restart runtime, then run from
section 2 onwards. Later runs in the same session can skip this.

In [ ]:
!pip install -qU --force-reinstall Pillow
!pip install -qU transformers accelerate bert-score scipy
print()
print('=' * 60)
print('FIRST RUN ON FRESH RUNTIME: Runtime -> Restart runtime now,')
print('then run cells from sec 2 onwards (do NOT re-run sec 1).')
print('=' * 60)


## 2. Unzip dataset

Unpacks `images.zip` and `annotations.zip` into `/content/`. Idempotent, safe to
re-run.

In [ ]:
import os
import zipfile

CONTENT = '/content'
ARCHIVES = [
    ('images.zip', 'images'),
    ('annotations.zip', 'annotations'),
]

for zip_name, folder_name in ARCHIVES:
    zip_path = os.path.join(CONTENT, zip_name)
    dest = os.path.join(CONTENT, folder_name)
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f'Missing {zip_path!r}. Upload {zip_name} to /content/.')
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest)
    print(f'OK: {zip_name} -> {dest}/')

for p in (
    os.path.join(CONTENT, 'annotations', 'evaluation_prompts.json'),
    os.path.join(CONTENT, 'annotations', 'ground_truth.json'),
):
    if not os.path.isfile(p):
        raise FileNotFoundError(f'Expected file missing after unzip: {p}')

for sub in ('stairs', 'elevators', 'facility_signs'):
    d = os.path.join(CONTENT, 'images', sub)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.lower().endswith('.jpg')])
        print(f'  images/{sub}/: {n} .jpg files')
    else:
        print(f'  WARNING: missing folder {d}')

print('Unzip complete.')


## 3. Configuration

Paths, decoding protocol, seeds and latency-benchmark parameters. Greedy decoding
(`do_sample=False`, `num_beams=1`) makes the run deterministic across model families.

In [ ]:
import random
import numpy as np
import torch

HF_TOKEN = ''  # paste your HuggingFace read token here (PaliGemma and Gemma weights are gated)

IMAGES_PATH      = '/content/images'
ANNOTATIONS_PATH = '/content/annotations'
MODELS_JSON_PATH = '/content/models.json'
RESULTS_PATH     = '/content/results/exp1_update'

MAX_NEW_TOKENS = 30

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DECODING_PROTOCOL = {
    'do_sample': False,
    'num_beams': 1,
    'temperature': None,
    'max_new_tokens': MAX_NEW_TOKENS,
    'enable_thinking': False,
    'torch_dtype': 'auto on CUDA (bfloat16 for Gemma, bfloat16 for PaliGemma); float32 on CPU',
    'gemma_recommended_overridden': 'stochastic (T=1.0, top_p=0.95, top_k=64) replaced by greedy for reproducibility',
}

N_WARMUP = 3
N_LATENCY_RUNS = 3

os.makedirs(RESULTS_PATH, exist_ok=True)

print('Config OK')
print(f'Images:      {IMAGES_PATH}')
print(f'Annotations: {ANNOTATIONS_PATH}')
print(f'Results:     {RESULTS_PATH}')
print(f'MAX_NEW_TOKENS: {MAX_NEW_TOKENS}')
print(f'Seed: {SEED}  |  cudnn deterministic: True')
print(f'Latency protocol: N_WARMUP={N_WARMUP}, N_LATENCY_RUNS={N_LATENCY_RUNS}')


## 4. Reproducibility card

Written to `exp1_reproducibility.json`: library versions, GPU, decoding and latency
protocol, dataset SHA-256 hashes and the list of models evaluated.

In [ ]:
import subprocess
import datetime
import json
import hashlib
import transformers
from importlib import metadata as importlib_metadata


def get_gpu_name():
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return 'CPU'


def get_git_commit():
    try:
        return subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return 'unavailable'


def _pkg_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return 'unavailable'


def _file_sha256(path, chunk_size=65536):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


with open(MODELS_JSON_PATH) as f:
    models_config = json.load(f)

run_models = models_config['models']

DATASET_ARTIFACT_PATHS = [
    '/content/images.zip',
    '/content/annotations/ground_truth.json',
    '/content/annotations/evaluation_prompts.json',
    '/content/models.json',
]
dataset_artifact_hashes = {}
for p in DATASET_ARTIFACT_PATHS:
    if os.path.isfile(p):
        dataset_artifact_hashes[p] = {
            'sha256': _file_sha256(p),
            'size_bytes': os.path.getsize(p),
        }

LATENCY_PROTOCOL = {
    'n_warmup': N_WARMUP,
    'n_latency_runs': N_LATENCY_RUNS,
    'cuda_synchronize_around_timer': True,
    'warmup_image_source': 'synthetic 224x224 deterministic RGB noise, seed=SEED',
    'image_order': 'shuffled deterministically per run (seed = SEED*1000 + run_idx)',
}

repro = {
    'experiment': 'exp1_update',
    'date': datetime.datetime.now().isoformat(timespec='seconds'),
    'gpu': get_gpu_name(),
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'pillow_version': _pkg_version('pillow'),
    'bert_score_version': _pkg_version('bert-score'),
    'seed': SEED,
    'cudnn_deterministic': True,
    'max_new_tokens': MAX_NEW_TOKENS,
    'images_path': IMAGES_PATH,
    'annotations_path': ANNOTATIONS_PATH,
    'git_commit': get_git_commit(),
    'models_evaluated': [m['model_id'] for m in run_models],
    'decoding_protocol': DECODING_PROTOCOL,
    'latency_protocol': LATENCY_PROTOCOL,
    'dataset_artifact_hashes': dataset_artifact_hashes,
}

repro_path = f'{RESULTS_PATH}/exp1_reproducibility.json'
with open(repro_path, 'w') as f:
    json.dump(repro, f, indent=2)

print('REPRODUCIBILITY CARD')
print('=' * 40)
preview = {k: v for k, v in repro.items() if k != 'dataset_artifact_hashes'}
preview['dataset_artifact_hashes'] = f'<{len(dataset_artifact_hashes)} SHA-256 fingerprints saved to JSON>'
for k, v in preview.items():
    print(f'  {k}: {v}')


## 5. Imports and dataset loading

Loads the Experiment 1 questions from `evaluation_prompts.json` and the per-image expected
answers from `ground_truth.json`, then builds the task list applying the category
filters declared in the questions schema.

In [ ]:
import gc
import re
import time
from collections import defaultdict
from PIL import Image
import pandas as pd
from transformers import AutoModelForCausalLM, AutoProcessor, PaliGemmaForConditionalGeneration
from bert_score import score as bertscore_fn

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

with open(os.path.join(ANNOTATIONS_PATH, 'evaluation_prompts.json')) as f:
    questions_payload = json.load(f)
with open(os.path.join(ANNOTATIONS_PATH, 'ground_truth.json')) as f:
    ground_truth = json.load(f)

exp1_questions = questions_payload['experiment1']['questions']

print(f'Exp1 questions: {len(exp1_questions)}')
for q in exp1_questions:
    print(f'  - {q["id"]}: applies_to={q["applies_to_categories"]}, task_type={q["task_type"]}')
print(f'Ground truth entries: {len(ground_truth)}')


## 6. Metric primitives

Pure functions that compute the metrics declared in `evaluation_prompts.json`: each takes
ground-truth and prediction lists and returns a scalar or a structured dict.

In [ ]:
def normalize(s):
    """Lowercase, strip surrounding whitespace, drop punctuation, collapse internal whitespace.

    Handles None (returns ''). Strips punctuation so 'Up.' and 'up' match.
    Internal whitespace collapsed to single spaces. Standard VQA-evaluation normalization.
    """
    if s is None:
        return ''
    s = str(s).lower().strip()
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def accuracy(y_true, y_pred):
    if not y_true:
        return 0.0
    return sum(1 for t, p in zip(y_true, y_pred) if normalize(p) == normalize(t)) / len(y_true)


def per_class_prf(y_true, y_pred, classes):
    """Per-class precision/recall/F1. Returns dict keyed by class."""
    out = {}
    for c in classes:
        c_norm = normalize(c)
        tp = sum(1 for t, p in zip(y_true, y_pred) if normalize(t) == c_norm and normalize(p) == c_norm)
        fp = sum(1 for t, p in zip(y_true, y_pred) if normalize(t) != c_norm and normalize(p) == c_norm)
        fn = sum(1 for t, p in zip(y_true, y_pred) if normalize(t) == c_norm and normalize(p) != c_norm)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        out[c] = {
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'f1': round(f1, 4),
            'support': sum(1 for t in y_true if normalize(t) == c_norm),
            'tp': tp, 'fp': fp, 'fn': fn,
        }
    return out


def macro_prf(y_true, y_pred, classes):
    """Macro-averaged precision/recall/F1 over the given classes (with support>0)."""
    per_c = per_class_prf(y_true, y_pred, classes)
    supported = [c for c in classes if per_c[c]['support'] > 0]
    if not supported:
        return {'macro_precision': 0.0, 'macro_recall': 0.0, 'macro_f1': 0.0, 'classes_used': []}
    p_macro = sum(per_c[c]['precision'] for c in supported) / len(supported)
    r_macro = sum(per_c[c]['recall'] for c in supported) / len(supported)
    f1_macro = sum(per_c[c]['f1'] for c in supported) / len(supported)
    return {
        'macro_precision': round(p_macro, 4),
        'macro_recall': round(r_macro, 4),
        'macro_f1': round(f1_macro, 4),
        'classes_used': supported,
    }


def confusion_matrix(y_true, y_pred, classes):
    """Returns a {true_class: {pred_class: count}} dict. Includes 'other' for unrecognised predictions."""
    classes_norm = [normalize(c) for c in classes]
    mat = {c: {c2: 0 for c2 in classes + ['other']} for c in classes}
    for t, p in zip(y_true, y_pred):
        tn, pn = normalize(t), normalize(p)
        if tn not in classes_norm:
            continue  # skip rows for unsupported true class
        true_label = classes[classes_norm.index(tn)]
        if pn in classes_norm:
            pred_label = classes[classes_norm.index(pn)]
        else:
            pred_label = 'other'
        mat[true_label][pred_label] += 1
    return mat


def keyword_recall(predictions, ground_truth_elements, expected_keywords):
    """For each prediction, check if any of its class's expected keywords appears in the response.
    Used only for element_identification.
    Returns: scalar in [0, 1] = fraction of correct keyword recalls.
    """
    if not predictions:
        return 0.0
    hits = 0
    for pred, gt_element in zip(predictions, ground_truth_elements):
        if gt_element not in expected_keywords:
            continue  # element not in our keyword dict, skip from numerator
        keywords_for_class = expected_keywords[gt_element]
        pred_norm = normalize(pred)
        for kw in keywords_for_class:
            if re.search(r'\b' + re.escape(normalize(kw)) + r'\b', pred_norm):
                hits += 1
                break
    # Denominator: count entries where the gt_element has keywords defined
    denom = sum(1 for g in ground_truth_elements if g in expected_keywords)
    return round(hits / denom, 4) if denom > 0 else 0.0


def avg_output_length_words(predictions):
    if not predictions:
        return 0.0
    return round(sum(len(p.split()) for p in predictions) / len(predictions), 2)


print('Metric primitives defined')


## 7. Model loading (family dispatch)

Dispatches on the `family` field in `models.json`. PaliGemma uses
`PaliGemmaForConditionalGeneration` with a raw `<image>{question}` prompt; Gemma 4
uses `AutoModelForCausalLM` with `apply_chat_template` and a structured messages list.

In [ ]:
def make_synthetic_image(seed=None):
    """Deterministic 224x224 RGB-noise PIL image for warmup."""
    rng = np.random.default_rng(SEED if seed is None else seed)
    arr = rng.integers(0, 256, size=(224, 224, 3), dtype=np.uint8)
    return Image.fromarray(arr, mode='RGB')


def load_model(meta):
    """Dispatch loader based on model family. Returns (processor, model, family)."""
    family = meta.get('family', 'paligemma')
    model_id = meta['model_id']

    if family == 'paligemma':
        dtype = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
        processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
        model = PaliGemmaForConditionalGeneration.from_pretrained(
            model_id, token=HF_TOKEN,
            torch_dtype=dtype,
            device_map='auto' if DEVICE == 'cuda' else None,
        )
        if DEVICE != 'cuda':
            model = model.to(DEVICE)
        model.eval()
        return processor, model, family

    elif family == 'gemma4':
        processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, token=HF_TOKEN,
            dtype='auto' if DEVICE == 'cuda' else torch.float32,
            device_map='auto' if DEVICE == 'cuda' else None,
        )
        if DEVICE != 'cuda':
            model = model.to(DEVICE)
        model.eval()
        return processor, model, family

    else:
        raise ValueError(f'Unknown family {family!r} for model {model_id}')


print('Model loader defined')


## 8. Inference primitives

`run_inference_timed` is the single entry point; internally it dispatches to the
PaliGemma or Gemma 4 flow. Greedy decoding is applied uniformly, and timing is
CUDA-synchronised to match the reproducibility protocol.

In [ ]:
GEMMA_SYSTEM_PROMPT = (
    'You are a visual assistant for indoor navigation. '
    'Answer concisely and exactly in the format requested by the question. '
    'Do not add explanations or extra words.'
)


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def build_paligemma_inputs(processor, image, question, model):
    prompt = f'<image>{question}'
    inputs = processor(text=prompt, images=image, padding='longest',
                       do_convert_rgb=True, return_tensors='pt').to(model.device)
    inputs = inputs.to(dtype=model.dtype)
    return inputs


def build_gemma_inputs(processor, image, question, model):
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': GEMMA_SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': question},
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        enable_thinking=False,
    ).to(model.device)
    return inputs.to(dtype=model.dtype)


def decode_response(processor, inputs, outputs):
    input_len = inputs['input_ids'].shape[-1]
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def run_inference_timed(processor, model, family, image, question):
    """Family-dispatched inference. Returns (prediction, latency_ms).
    Decoding kwargs match DECODING_PROTOCOL exactly.
    """
    if family == 'paligemma':
        inputs = build_paligemma_inputs(processor, image, question, model)
    elif family == 'gemma4':
        inputs = build_gemma_inputs(processor, image, question, model)
    else:
        raise ValueError(f'Unknown family {family!r}')

    _sync()
    t0 = time.time()
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )
    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    _sync()
    latency_ms = (time.time() - t0) * 1000.0
    return decode_response(processor, inputs, output), latency_ms


print('Inference primitives defined')


## 9. Task list builder

For each question and each image whose category matches `applies_to_categories`
(and passes the optional field filter), produces a task with the image path,
category, question and expected answer.

In [ ]:
def build_task_list(questions, ground_truth, images_path):
    tasks = []
    for q in questions:
        applies = set(q['applies_to_categories'])
        applies_field = q.get('applies_only_when_field_equals', None)
        for img_rel, gt in ground_truth.items():
            if gt['category'] not in applies:
                continue
            if applies_field is not None:
                ok = all(gt.get(k) == v for k, v in applies_field.items())
                if not ok:
                    continue
            expected = gt.get(q['expected_answer_field'])
            if expected is None:
                continue  # ground truth field missing for this image
            full_path = os.path.join(images_path, img_rel)
            if not os.path.isfile(full_path):
                print(f'  WARNING: missing image {full_path}')
                continue
            tasks.append({
                'image_rel': img_rel,
                'image_path': full_path,
                'category': gt['category'],
                'question_id': q['id'],
                'question_text': q['question_text'],
                'expected': expected,
                'task_type': q['task_type'],
                'ground_truth': gt,
            })
    return tasks


tasks_preview = build_task_list(exp1_questions, ground_truth, IMAGES_PATH)
print(f'Total Exp1 tasks: {len(tasks_preview)}')
by_q = defaultdict(int)
for t in tasks_preview:
    by_q[t['question_id']] += 1
for q_id, n in by_q.items():
    print(f'  {q_id}: {n} tasks')


## 10. Per-model evaluation pipeline

For one model: load, warm up, run `N_LATENCY_RUNS` timed passes over the task list
(predictions captured on the first run, latencies averaged across runs), compute
the per-question, state-recognition and operational metrics, save the artefacts
(`results.csv`, `aggregate.csv`, `efficiency.json`, `matrices.json`), and free GPU
memory.

In [ ]:
def get_question_def(q_id):
    for q in exp1_questions:
        if q['id'] == q_id:
            return q
    raise KeyError(q_id)


def compute_per_question_metrics(df, q_id):
    """Compute all metrics for one question's prediction set."""
    q_def = get_question_def(q_id)
    sub = df[df['question_id'] == q_id]
    if sub.empty:
        return {}

    y_true = sub['expected'].astype(str).tolist()
    y_pred = sub['prediction'].astype(str).tolist()

    out = {
        'question_id': q_id,
        'n': len(sub),
        'accuracy': round(accuracy(y_true, y_pred), 4),
        'avg_output_length_words': avg_output_length_words(y_pred),
    }

    # Determine class set for P/R/F1 computation
    if q_def['task_type'] == 'binary':
        classes = q_def['valid_classes']
    elif q_def['task_type'] == 'multiclass':
        classes = q_def.get('metric_classes_for_f1', [])
    else:
        classes = []

    if classes:
        m = macro_prf(y_true, y_pred, classes)
        out.update({
            'macro_precision': m['macro_precision'],
            'macro_recall':    m['macro_recall'],
            'macro_f1':        m['macro_f1'],
            'classes_used':    m['classes_used'],
        })

    # element_identification only: keyword_recall
    if q_id == 'element_identification':
        kw_dict = q_def['expected_keywords']
        out['keyword_recall'] = keyword_recall(y_pred, y_true, kw_dict)

    return out


def compute_state_aggregate(df):
    """Aggregate macro P/R/F1 across the 3 binary safety/sub-type tasks.
    Builds a unified 4-class problem (up/down/open/closed).
    """
    safety_qs = ['stairs_direction', 'elevator_state']
    sub = df[df['question_id'].isin(safety_qs)]
    if sub.empty:
        return {}
    y_true = sub['expected'].astype(str).tolist()
    y_pred = sub['prediction'].astype(str).tolist()
    classes = ['up', 'down', 'open', 'closed']
    m = macro_prf(y_true, y_pred, classes)
    return {
        'state_recognition_macro_precision': m['macro_precision'],
        'state_recognition_macro_recall':    m['macro_recall'],
        'state_recognition_macro_f1':        m['macro_f1'],
        'state_recognition_classes_used':    m['classes_used'],
        'state_recognition_n':               len(sub),
    }


def compute_confusion_matrices(df):
    """Per-question confusion matrices and per-class P/R/F1."""
    out = {}
    for q_id in df['question_id'].unique():
        q_def = get_question_def(q_id)
        sub = df[df['question_id'] == q_id]
        y_true = sub['expected'].astype(str).tolist()
        y_pred = sub['prediction'].astype(str).tolist()

        # full class set for confusion matrix (incl. excluded-from-F1 classes)
        if q_def['task_type'] == 'binary':
            cm_classes = q_def['valid_classes']
            f1_classes = q_def['valid_classes']
        else:
            cm_classes = q_def.get('metric_classes_for_f1', []) + q_def.get('metric_classes_excluded_from_f1', [])
            f1_classes = q_def.get('metric_classes_for_f1', [])

        cm = confusion_matrix(y_true, y_pred, cm_classes)
        per_c = per_class_prf(y_true, y_pred, f1_classes)
        out[q_id] = {
            'classes_in_matrix': cm_classes,
            'classes_in_f1': f1_classes,
            'matrix': cm,
            'per_class_metrics': per_c,
            'note': 'rows = ground truth, columns = predicted (incl. "other" for unrecognised predictions)',
        }
    return out


print('Aggregator helpers defined')


In [ ]:
def run_experiment1_for_model(meta):
    model_id = meta['model_id']
    family = meta.get('family', 'paligemma')
    safe_name = model_id.replace('/', '_').replace('-', '_')
    model_dir = f'{RESULTS_PATH}/{safe_name}'
    os.makedirs(model_dir, exist_ok=True)

    print(f'\n{"=" * 60}')
    print(f'MODEL: {model_id}  (family={family})')
    print(f'{"=" * 60}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t_load = time.time()
    processor, model, family = load_model(meta)
    model_load_time_s = round(time.time() - t_load, 2)
    n_parameters = sum(p.numel() for p in model.parameters())
    print(f'  Loaded on {DEVICE} - {n_parameters / 1e9:.2f}B params in {model_load_time_s:.1f}s')

    print(f'  Warming up ({N_WARMUP} iters on synthetic image, discarded)...')
    warmup_img = make_synthetic_image()
    for _ in range(N_WARMUP):
        run_inference_timed(processor, model, family, warmup_img, 'What is in this image?')

    tasks = build_task_list(exp1_questions, ground_truth, IMAGES_PATH)
    n_tasks = len(tasks)
    print(f'  Tasks to run: {n_tasks} per latency run x {N_LATENCY_RUNS} runs')

    predictions = [None] * n_tasks
    latencies = [[] for _ in range(n_tasks)]
    t_total = time.time()

    for run_idx in range(N_LATENCY_RUNS):
        order = np.arange(n_tasks)
        np.random.default_rng(SEED * 1000 + run_idx).shuffle(order)
        for task_idx in order:
            task = tasks[task_idx]
            image = Image.open(task['image_path']).convert('RGB')
            try:
                pred, lat_ms = run_inference_timed(processor, model, family, image, task['question_text'])
            except Exception as e:
                print(f'    [FAIL] run {run_idx + 1} task {task_idx}: {e}')
                pred, lat_ms = '', 0.0
            latencies[task_idx].append(lat_ms)
            if run_idx == 0:
                predictions[task_idx] = pred

    total_time_s = time.time() - t_total

    rows = []
    for t, pred, lats in zip(tasks, predictions, latencies):
        lat_mean = float(np.mean(lats)) if lats else 0.0
        lat_std = float(np.std(lats, ddof=1)) if len(lats) >= 2 else 0.0
        rows.append({
            'image': t['image_rel'],
            'category': t['category'],
            'question_id': t['question_id'],
            'expected': t['expected'],
            'prediction': pred,
            'correct': normalize(pred) == normalize(t['expected']),
            'inference_time_ms': round(lat_mean, 2),
            'latency_ms_std': round(lat_std, 2),
        })
        for i, lat in enumerate(lats):
            rows[-1][f'latency_ms_run{i}'] = round(lat, 2)
    df = pd.DataFrame(rows)

    print(f'  Inferences: {len(df)} unique tasks, {len(df) * N_LATENCY_RUNS} timed calls in {total_time_s:.1f}s')

    # Per-question metrics
    agg_rows = []
    for q_id in df['question_id'].unique():
        agg_rows.append(compute_per_question_metrics(df, q_id))
    agg_df = pd.DataFrame(agg_rows)

    # Aggregated state-recognition metrics + overall
    state_metrics = compute_state_aggregate(df)
    overall_acc = round(df['correct'].mean(), 4) if not df.empty else 0.0

    # Confusion matrices + per-class
    matrices = compute_confusion_matrices(df)

    vram_peak = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
    total_inf_ms = df['inference_time_ms'].sum()
    avg_inf_ms = float(df['inference_time_ms'].mean()) if not df.empty else 0.0
    avg_inf_ms_std = float(df['latency_ms_std'].mean()) if not df.empty else 0.0

    efficiency = {
        'model_id': model_id,
        'family': family,
        'experiment': 'exp1_update',
        'n_parameters': n_parameters,
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'n_inferences': len(df),
        'overall_accuracy': overall_acc,
        **state_metrics,
        'vram_peak_gb': round(vram_peak, 3),
        'model_load_time_s': model_load_time_s,
        'total_wall_time_s': round(total_time_s, 2),
        'mean_inference_time_ms': round(avg_inf_ms, 2),
        'avg_inference_time_ms_std': round(avg_inf_ms_std, 2),
        'throughput_inf_per_s': round(len(df) / (total_inf_ms / 1000), 3) if total_inf_ms > 0 else 0,
    }

    matrices_payload = {
        'model_id': model_id,
        'family': family,
        'experiment': 'exp1_update',
        'questions': matrices,
    }

    df.to_csv(f'{model_dir}/results.csv', index=False)
    agg_df.to_csv(f'{model_dir}/aggregate.csv', index=False)
    with open(f'{model_dir}/efficiency.json', 'w') as f:
        json.dump(efficiency, f, indent=2)
    with open(f'{model_dir}/matrices.json', 'w') as f:
        json.dump(matrices_payload, f, indent=2)

    print(f'  All files saved to {model_dir}')

    summary = {
        'model_id': model_id,
        'model_safe': safe_name,
        'family': family,
        'version': meta.get('version', ''),
        'size': meta.get('size', ''),
        'type': meta.get('type', ''),
        'resolution': meta.get('resolution', ''),
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'overall_accuracy': overall_acc,
        **{k: v for k, v in state_metrics.items() if k != 'state_recognition_classes_used'},
        'vram_peak_gb': efficiency['vram_peak_gb'],
        'model_load_time_s': model_load_time_s,
        'mean_inference_time_ms': efficiency['mean_inference_time_ms'],
        'avg_inference_time_ms_std': efficiency['avg_inference_time_ms_std'],
        'throughput_inf_per_s': efficiency['throughput_inf_per_s'],
    }
    # Per-question metrics for the cross-model summary
    # Includes accuracy + macro_precision + macro_recall + macro_f1 per question,
    # plus keyword_recall and avg_output_length where applicable.
    for _, row in agg_df.iterrows():
        q_id = row['question_id']
        summary[f'{q_id}_accuracy'] = row.get('accuracy')
        for m in ('macro_precision', 'macro_recall', 'macro_f1'):
            if m in row and pd.notna(row[m]):
                summary[f'{q_id}_{m}'] = row[m]
        if 'avg_output_length_words' in row and pd.notna(row['avg_output_length_words']):
            summary[f'{q_id}_avg_output_length_words'] = row['avg_output_length_words']
        if q_id == 'element_identification' and 'keyword_recall' in row:
            summary['element_identification_keyword_recall'] = row['keyword_recall']

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('  GPU memory freed')
    return summary


print('Orchestrator defined')


## 11. Main loop

Runs every model in `models.json`. A model that crashes is recorded in
`failed_models` instead of aborting the batch.

In [ ]:
all_summaries = []
failed_models = []

for meta in run_models:
    model_id = meta['model_id']
    try:
        summary = run_experiment1_for_model(meta)
        all_summaries.append(summary)
        print(f'  OK: {model_id}  overall_accuracy={summary["overall_accuracy"]}')
    except Exception as exc:
        print(f'  FAILED: {model_id} - {exc}')
        failed_models.append({'model_id': model_id, 'error': str(exc)})
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f'\nCompleted: {len(all_summaries)}/{len(run_models)} models')
if failed_models:
    print('Failed:', [m['model_id'] for m in failed_models])
    with open(f'{RESULTS_PATH}/failed_models.json', 'w') as f:
        json.dump(failed_models, f, indent=2)


## 12. Cross-model comparison table

Key metrics side by side for every model that finished, saved to
`all_models_summary.csv`.

In [ ]:
comp_df = pd.DataFrame(all_summaries)

# All available metrics, sorted into logical groups for readability
headline_cols = [
    # identity
    'model_id', 'family', 'version', 'size', 'type', 'resolution', 'n_parameters_billions',
    # global
    'overall_accuracy',
    # state recognition (cross-task aggregate over the 3 binary state tasks)
    'state_recognition_macro_precision', 'state_recognition_macro_recall', 'state_recognition_macro_f1',
    # element_identification (multi-class, 3 classes)
    'element_identification_accuracy', 'element_identification_macro_precision',
    'element_identification_macro_recall', 'element_identification_macro_f1',
    'element_identification_keyword_recall', 'element_identification_avg_output_length_words',
    # stairs_direction (binary)
    'stairs_direction_accuracy', 'stairs_direction_macro_precision',
    'stairs_direction_macro_recall', 'stairs_direction_macro_f1',
    'stairs_direction_avg_output_length_words',
    # elevator_state (binary)
    'elevator_state_accuracy', 'elevator_state_macro_precision',
    'elevator_state_macro_recall', 'elevator_state_macro_f1',
    'elevator_state_avg_output_length_words',
    # operational
    'vram_peak_gb', 'model_load_time_s',
    'mean_inference_time_ms', 'avg_inference_time_ms_std', 'throughput_inf_per_s',
]
headline_cols = [c for c in headline_cols if c in comp_df.columns]
summary_df = comp_df[headline_cols].copy()

summary_df.to_csv(f'{RESULTS_PATH}/all_models_summary.csv', index=False)
print(f'Saved: {RESULTS_PATH}/all_models_summary.csv')
print()
print('ALL MODELS - SUMMARY')
print('=' * 80)
display(summary_df.round(4))


## 13. Console summary

The comparison table is already saved in section 12. This cell prints the best
model per headline metric.

In [ ]:
if not comp_df.empty:
    print('=== EXPERIMENT 1 - BEST MODEL PER METRIC ===')
    for metric in ['overall_accuracy', 'state_recognition_macro_f1',
                   'element_identification_macro_f1', 'element_identification_keyword_recall',
                   'stairs_direction_macro_f1', 'elevator_state_macro_f1']:
        if metric in comp_df.columns:
            s = comp_df[metric].dropna()
            if not s.empty:
                idx = s.idxmax()
                print(f'  {metric}: {comp_df.loc[idx, "model_id"]} ({s.max():.4f})')

    print()
    print('=== FASTEST (lowest mean_inference_time_ms) ===')
    s = comp_df['mean_inference_time_ms'].dropna()
    if not s.empty:
        idx = s.idxmin()
        print(f'  {comp_df.loc[idx, "model_id"]} ({s.min():.1f} ms)')
else:
    print('No models completed successfully.')


## 14. Download results as ZIP

Packages everything under the results path into a single ZIP and triggers a browser
download. Run it last so the saved file reflects the finished experiment.

In [ ]:
import shutil
from google.colab import files

zip_base = '/content/exp1_update_results'
shutil.make_archive(zip_base, 'zip', RESULTS_PATH)
zip_path = zip_base + '.zip'
print(f'Created {zip_path}  ({os.path.getsize(zip_path) / 1024**2:.2f} MB)')
files.download(zip_path)
